In [ ]:
import queue
import time

import numpy as np
import tritonclient.grpc as grpcclient
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TRITON_URL = "localhost:8001"
MODEL_NAME = "easymp"
SAMPLE_RATE = 22050  # codec output_sample_rate

In [ ]:
def synthesize(text: str) -> np.ndarray:
    """Send a TTS request and collect streamed audio chunks."""
    result_q: queue.Queue = queue.Queue()

    def _on_response(result, error):
        result_q.put((result, error))

    client = grpcclient.InferenceServerClient(url=TRITON_URL)
    client.start_stream(callback=_on_response)

    text_input = grpcclient.InferInput("text", [1, 1], "BYTES")
    text_input.set_data_from_numpy(np.array([[text]], dtype=object))

    client.async_stream_infer(
        model_name=MODEL_NAME,
        inputs=[text_input],
        outputs=[grpcclient.InferRequestedOutput("audio")],
    )

    chunks = []
    t0 = time.perf_counter()
    t_first = None

    while True:
        result, error = result_q.get(timeout=120)
        if error:
            client.stop_stream()
            raise RuntimeError(str(error))

        audio = result.as_numpy("audio").squeeze()
        if audio.size > 0:
            if t_first is None:
                t_first = time.perf_counter()
            chunks.append(audio)

        response = result.get_response()
        final_param = response.parameters.get("triton_final_response")
        if final_param and getattr(final_param, "bool_param", False):
            break

    client.stop_stream()

    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first else elapsed
    print(f"Received {len(chunks)} chunks | TTFA: {ttfa:.3f}s | total: {elapsed:.3f}s")
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)

In [ ]:
audio = synthesize(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
)
print(f"Got {len(audio)} samples — {len(audio) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz")

In [ ]:
t = np.arange(len(audio)) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("EasyMP Waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=SAMPLE_RATE))

## Streaming-text input

Instead of one request with the full `text`, push subword ids a few at a time over
a single gRPC stream. Every chunk shares a `stream_id`; mark the first request with
`stream_start=true` (it also carries `speaker` / `context_text`) and the last with
`stream_end=true`. The server appends text-EOS + a free-running acoustic tail, and
streams **all** audio back on the `stream_start` request's response (follow-up chunk
requests just feed tokens and return an empty final response). This is the
live-client path: synthesis begins before the whole text is known.

In [ ]:
import uuid

from transformers import AutoTokenizer

# Same tokenizer the converted model bundles; used to turn text into the subword
# ids the streaming path expects (specials off — the server appends text-EOS).
MODEL_DIR = "/home/vklimkov/workspace/emp/NeMo/examples/tts/easymagpie_vllm_omni/easymp_vllm_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)


def _str_input(name: str, value: str):
    t = grpcclient.InferInput(name, [1, 1], "BYTES")
    t.set_data_from_numpy(np.array([[value]], dtype=object))
    return t


def _bool_input(name: str, value: bool):
    t = grpcclient.InferInput(name, [1, 1], "BOOL")
    t.set_data_from_numpy(np.array([[value]], dtype=bool))
    return t


def _token_input(tokens):
    t = grpcclient.InferInput("text_token", [1, len(tokens)], "INT32")
    t.set_data_from_numpy(np.array([tokens], dtype=np.int32))
    return t


def synthesize_streaming(
    text: str,
    speaker: str = "eng",
    context_text: str = "[EN]",
    tokens_per_chunk: int = 1,
    delay_s: float = 0.0,
) -> np.ndarray:
    """Stream subword ids ``tokens_per_chunk`` at a time; collect streamed audio.

    ``delay_s`` (optional) sleeps between chunks to mimic tokens trickling in from
    an upstream producer. Audio arrives on the ``stream_start`` request's response.
    """
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = [token_ids[i : i + tokens_per_chunk] for i in range(0, len(token_ids), tokens_per_chunk)] or [[]]

    stream_id = uuid.uuid4().hex
    start_rid = f"{stream_id}-start"
    result_q: queue.Queue = queue.Queue()

    client = grpcclient.InferenceServerClient(url=TRITON_URL)
    client.start_stream(callback=lambda result, error: result_q.put((result, error)))

    def _send(inputs, request_id):
        client.async_stream_infer(
            model_name=MODEL_NAME,
            inputs=inputs,
            outputs=[grpcclient.InferRequestedOutput("audio")],
            request_id=request_id,
        )

    # stream_start: speaker/context (+ first token chunk). stream_end rides along if
    # the whole utterance is a single chunk.
    start_inputs = [
        _str_input("stream_id", stream_id),
        _bool_input("stream_start", True),
        _str_input("speaker", speaker),
        _str_input("context_text", context_text),
    ]
    if chunks[0]:
        start_inputs.append(_token_input(chunks[0]))
    if len(chunks) == 1:
        start_inputs.append(_bool_input("stream_end", True))
    _send(start_inputs, start_rid)

    for ci, chunk in enumerate(chunks[1:], start=1):
        if delay_s:
            time.sleep(delay_s)
        inputs = [_str_input("stream_id", stream_id), _token_input(chunk)]
        if ci == len(chunks) - 1:
            inputs.append(_bool_input("stream_end", True))
        _send(inputs, f"{stream_id}-c{ci}")

    audio_chunks = []
    t0 = time.perf_counter()
    t_first = None
    while True:
        result, error = result_q.get(timeout=120)
        if error:
            client.stop_stream()
            raise RuntimeError(str(error))
        response = result.get_response()
        # Audio only ever rides on the stream_start request's response.
        if response.id != start_rid:
            continue
        audio = result.as_numpy("audio")
        if audio is not None:
            audio = audio.squeeze()
            if audio.size > 0:
                if t_first is None:
                    t_first = time.perf_counter()
                audio_chunks.append(audio)
        final_param = response.parameters.get("triton_final_response")
        if final_param and getattr(final_param, "bool_param", False):
            break

    client.stop_stream()
    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first else elapsed
    print(f"Streamed {len(chunks)} text chunks | {len(audio_chunks)} audio chunks | TTFA: {ttfa:.3f}s | total: {elapsed:.3f}s")
    return np.concatenate(audio_chunks) if audio_chunks else np.array([], dtype=np.float32)

In [ ]:
audio_stream = synthesize_streaming(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the unicorns to fly.",
    tokens_per_chunk=1,
    delay_s=0.003,  # set e.g. 0.02 to simulate tokens trickling in from an upstream LLM
)
print(f"Got {len(audio_stream)} samples — {len(audio_stream) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz")
display(Audio(audio_stream, rate=SAMPLE_RATE))